In [ ]:
#Load data
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data" / "raw"

users = pd.read_csv(DATA_DIR / "users.csv", parse_dates=["install_date"])
sessions = pd.read_csv(DATA_DIR / "sessions.csv", parse_dates=["session_start_ts", "session_end_ts"])
events = pd.read_csv(DATA_DIR / "economy_events.csv", parse_dates=["event_ts"])
items = pd.read_csv(DATA_DIR / "items.csv")

In [ ]:
#Helper date columns
sessions["session_date"] = sessions["session_start_ts"].dt.date
events["event_date"] = events["event_ts"].dt.date

In [ ]:
# DAU
dau = (
    sessions.groupby("session_date")["user_id"]
    .nunique()
    .rename("dau")
    .reset_index()
)

# Revenue & payers
rev_daily = (
    events.query("event_type == 'iap_purchase'")
          .groupby("event_date")
          .agg(total_revenue=("revenue_usd", "sum"),
               payers=("user_id", "nunique"))
          .reset_index()
          .rename(columns={"event_date": "date"})
)

dau = dau.rename(columns={"session_date": "date"})
daily_kpis = dau.merge(rev_daily, on="date", how="left").fillna({"total_revenue": 0, "payers": 0})

daily_kpis["arpu"] = daily_kpis["total_revenue"] / daily_kpis["dau"]
daily_kpis["arppu"] = daily_kpis["total_revenue"] / daily_kpis["payers"].replace(0, pd.NA)

daily_kpis.head()

In [ ]:
#Save to data/processed/daily_kpis.csv for reuse
OUT_DIR = Path("..") / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
daily_kpis.to_csv(OUT_DIR / "daily_kpis.csv", index=False)

In [ ]:
#Plot ARPU / Revenue Trends
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots()

ax1.plot(daily_kpis["date"], daily_kpis["total_revenue"], label="Total Revenue")
ax1.set_xlabel("Date")
ax1.set_ylabel("Revenue (USD)")

ax2 = ax1.twinx()
ax2.plot(daily_kpis["date"], daily_kpis["arpu"], linestyle="--", label="ARPU")

fig.autofmt_xdate()
plt.title("Daily Revenue and ARPU")
plt.show()

In [ ]:
#Retention Cohorts in Python

# Install date per user
cohort = users[["user_id", "install_date"]].rename(columns={"install_date": "cohort_date"})

# User active dates
activity = (
    sessions[["user_id", "session_date"]]
    .drop_duplicates()
    .rename(columns={"session_date": "activity_date"})
)

df = cohort.merge(activity, on="user_id", how="left")
df["days_since_install"] = (pd.to_datetime(df["activity_date"]) - df["cohort_date"]).dt.days

# Pivot into cohort matrix
cohort_pivot = (
    df[df["days_since_install"].between(0, 30)]
      .groupby(["cohort_date", "days_since_install"])["user_id"]
      .nunique()
      .reset_index()
)

# Size of each cohort (day 0 installs)
cohort_sizes = cohort_pivot[cohort_pivot["days_since_install"] == 0][["cohort_date", "user_id"]]
cohort_sizes = cohort_sizes.rename(columns={"user_id": "cohort_size"})

cohort_pivot = cohort_pivot.merge(cohort_sizes, on="cohort_date")
cohort_pivot["retention_rate"] = cohort_pivot["user_id"] / cohort_pivot["cohort_size"]

# For your report, you might subset to [1,7,30]
ret_summary = cohort_pivot[cohort_pivot["days_since_install"].isin([1,7,30])]
ret_summary.head()

In [ ]:
#Funnel
import numpy as np

installed = set(users["user_id"])
sessioned = set(sessions["user_id"])
spent_soft = set(events.query("event_type == 'spend_soft'")["user_id"])
payers = set(events.query("event_type == 'iap_purchase'")["user_id"])

funnel = [
    ("Installed", len(installed)),
    ("Any session", len(sessioned)),
    ("Spent soft", len(spent_soft)),
    ("Payer", len(payers)),
]

funnel